# Suite GEN — Sampling and generators

Samples are specified exactly on the random source's 32-bit words, so both implementations draw the same values from the
same seed. `Generate` streams new objects drawn from a distribution: each step chooses a case by weight, and builds what
the case's predicate requires by equality.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Distributions as D, Generators as G, Predicates as P, Queries as Q, Sampling as M, Validators as V } from "@mbse/patterns";
import { Errors, JSON as SchemaJSON, Stores } from "@mbse/schemas/Framework";
import { Contact, Directory, assert, book, equal, listed, raises, same } from "./support.js";

/** A random source of the given words, for exact cases. */
class Words implements Stores.Random {
  readonly words: bigint[];

  constructor(...words: bigint[]) {
    this.words = words;
  }

  next_u32(): bigint {
    return this.words.shift() as bigint;
  }

  split(_key: string): Stores.Random {
    return this;
  }
}

function take<T>(items: Iterator<T>, n: number): T[] {
  return Array.from({ length: n }, () => items.next().value as T);
}

const APerson = { person: Contact };
const HasName = new P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters((p) => p.name("name"))
  .requires(E.variable("person").name.eq(E.variable("name"))).create();
const named = (weight: number, name: string) => (wt: D.OfCase.Builder) => wt.weight(weight).requires(
  (pred) => pred.symbols(APerson).requires(HasName.call(pred.person, name)));
const Names = new D.OfWeights.Builder().name("Names").symbols(APerson).decreasing(
  named(10, "alice"), named(5, "ben"), named(15, "chermon"), named(7, "davi")).create();
const MAX = (1n << 32n) - 1n;

## GEN-01 · Samples are specified on the source's words

In [ ]:
{
  let r: Stores.Random = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.uniform(r)), [0.6303102186438938, 0.7270080560068604, 0.7486033647998483]));
  r = new Stores.PCG32(42n, 54n);
  assert(same(take({ next: () => ({ value: M.below(r, 6n), done: false }) }, 8), [3n, 3n, 2n, 1n, 1n, 4n, 5n, 3n]));
  r = new Stores.PCG32(42n, 54n);
  assert(same([0, 1, 2].map(() => M.below(r, 2n ** 40n)), [788047328265n, 208370070163n, 325543878766n])); // two words each
  r = new Stores.PCG32(42n, 54n);
  assert(same(Array.from({ length: 10 }, () => M.weighted(r, [10, 5, 15, 7])), [2, 2, 2, 2, 3, 0, 3, 0, 2, 2]));
  assert(M.uniform(new Words(0n, 0n)) === 0 && M.uniform(new Words(MAX, MAX)) === 1 - 2 ** -53); // the ends
  assert(M.below(new Words(MAX, 5n), 3n) === 2n); // 2**32 - 1 is past the last multiple of 3: rejected, then 5 % 3
  assert(M.below(new Words(7n), 1n) === 0n && M.weighted(new Words(0n, 0n), [0, 2, 0]) === 1);
  assert(M.weighted(new Words(MAX, MAX), [1, 2, 0]) === 1); // the last positive weight, at the top
  raises(Errors.ValueError, () => M.below(new Words(), 0n), "below needs a positive int, got 0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, -1]), "a weight must be a finite float, not negative, got -1.0");
  raises(Errors.ValueError, () => M.weighted(new Words(), [1, Infinity]), "a weight must be a finite float, not negative, got inf");
  raises(Errors.ValueError, () => M.weighted(new Words(), [0, 0]), "no weight is positive");
}

## GEN-02 · Generate chooses cases by weight and builds what their equalities require

In [ ]:
{
  const store = book();
  const generated = G.Generate(store, Names, new Stores.PCG32(42n));
  const first = take(generated, 12).map((m: any) => m.person.name);
  assert(same(first, ["chermon", "davi", "chermon", "davi", "alice", "ben", "alice", "davi", "alice", "ben", "chermon", "chermon"]));
  assert(same(take(G.Generate(book(), Names, new Stores.PCG32(42n)), 12).map((m: any) => m.person.name), first)); // the same seed
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(7n)), 12).map((m: any) => m.person.name), first));
  assert(!same(take(G.Generate(book(), Names, new Stores.PCG32(42n).split("other")), 12).map((m: any) => m.person.name), first));
  const names = new Map<string, number>();
  for (const m of take(generated, 3700) as any[]) names.set(m.person.name, (names.get(m.person.name) ?? 0) + 1);
  for (const [n, w] of [["alice", 10], ["ben", 5], ["chermon", 15], ["davi", 7]] as const) {
    assert(Math.abs((names.get(n) as number) - 100 * w) < 100, n);
  }
  const plain = (found: Map<string, Map<string, unknown>>) => [...found].map(([s, values]) => [s, [...values]]);
  assert(equal(plain(G.settings(Names.cases[0].predicate)), [["person", [["name", "alice"]]]]));
  const [person, age] = [E.variable("person"), E.variable("person").age];
  const nested = new P.OfPredicate.Builder().symbols(APerson).requires(E.literal(30n).eq(age).and_(HasName.call(person, "cy")))
    .requires(person.has("name")).create();
  assert(equal(plain(G.settings(nested)), [["person", [["age", 30n], ["name", "cy"]]]])); // either side of ==, through and and applications
  const ignored = new P.OfPredicate.Builder().symbols(APerson).parameters((p) => p.name("n")).requires(
    E.variable("x").name.eq("a")).requires(person.name.eq(E.variable("n"))).requires(person.name.eq(person.age)).requires(
    E.operation("get", person, 1n).eq("a")).requires(person.eq("a")).requires(HasName.call(person, E.variable("n"))).create();
  assert(G.settings(ignored).size === 0); // no symbol's property, or no literal value
}

## GEN-03 · A case its equalities cannot build is refused

In [ ]:
{
  const Grown = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(APerson).requires(pred.person.age.ge(18n)))).create();
  raises(Errors.ValueError, () => G.Generate(book(), Grown, new Stores.PCG32(42n)).next(),
    "case 0 cannot be generated from its equalities: what they build does not satisfy it");
  const Shadowed = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1e-9).requires((pred) => pred.symbols(APerson).requires(pred.person.has("name"))),
    named(1, "alice")).create();
  raises(Errors.ValueError, () => G.Generate(book(), Shadowed, new Stores.PCG32(42n)).next(),
    "case 1 cannot be generated from its equalities: what they build satisfies case 0");
  const Later = new D.OfWeights.Builder().symbols(APerson).decreasing(
    (wt) => wt.weight(1).requires((pred) => pred.symbols(APerson).requires(HasName.call(pred.person, "alice"))
      .requires(pred.person.age.ge(18n))),
    named(1e-9, "alice")).create();
  raises(Errors.ValueError, () => G.Generate(book(), Later, new Stores.PCG32(42n)).next(),
    "case 0 cannot be generated from its equalities: what they build does not satisfy it"); // a later case holds instead
  raises(Errors.ValueError, () => G.Generate(book(), new D.OfWeights.Builder().symbols(APerson).create(), new Stores.PCG32(42n)),
    "the distribution cannot be drawn from: a distribution needs a case");
}

## GEN-04 · Generated data is ordinary data: listed, validated, queried and written

In [ ]:
{
  const store = book();
  const people = take(G.Generate(store, Names, new Stores.PCG32(42n)), 6).map((m: any) => m.person);
  assert([...store.extent("Contact")].length === 0); // transient until linked to the store's data
  listed(store, ...people);
  assert([...store.extent("Contact")].length === 6);
  const NamedAlice = new P.OfPredicate.Builder().name("NamedAlice").symbols({ the: Contact })
    .requires(HasName.call(E.variable("the"), "alice")).create();
  const alices = [...new Q.Scan(store).select(NamedAlice)].map((m: any) => m.the);
  assert(same(alices, people.filter((p: any) => p.name === "alice")) && alices.length > 0);
  const problems = V.Validate(store, [NamedAlice]).Reachable(Directory, store.singleton("book.Directory"));
  assert(problems.length === 6 - alices.length && problems.every((p) => p.endsWith("'NamedAlice' does not hold")));
  assert(SchemaJSON.ToJSON(store).Reachable(Directory, store.singleton("book.Directory")).includes('"name": "chermon"'));
}